In [51]:
import pandas as pd
import numpy as np
from pathlib import Path

In [67]:
DATA_DIR = Path.cwd().parent / "data"
DATA_FILE = DATA_DIR / "superstore_orders.csv.gz"
SUBMISSION_DIR = Path.cwd().parent / "submission"
SUBMISSION_PROFITABILITY_FILE = SUBMISSION_DIR / "profitability_summary.csv"
SUBMISSION_DISCOUNT_FILE = SUBMISSION_DIR / "discount_summary.csv"
SUBMISSION_PRIORITY_FILE = SUBMISSION_DIR / "priority_segments.csv"

In [69]:
# LECTURA DE DATOS
datos = pd.read_csv(DATA_FILE, compression='gzip', encoding='utf-8', sep=";")

datos["Sales"] = datos["Sales"].str.replace(",", ".").astype(float)
datos["Profit"] = datos["Profit"].str.replace(",", ".").astype(float)
datos["Discount"] = datos["Discount"].str.replace(",", ".").astype(float)

In [ ]:
lines = len(datos['Order ID'])                                                   # Número de filas de los datos
orders = datos['Order ID'].nunique()                                             # Número de pedidos únicos  
sales = datos['Sales'].sum()                                                     # Cantidad de ventas totales
profit = datos["Profit"].sum()                                                   # Cantidad de ganancias totales
profit_margin = profit / sales                                                   # Margen de ganancia
datos["perdidas"] = datos["Profit"] < 0                                          # Identificar las filas con pérdidas
loss_lines = datos["perdidas"].sum()                                             # Total de líneas con pérdidas
loss_line_rate = datos["perdidas"].mean()                                        # Pérdida promedio por línea, sobre los productos que generan pérdida
lost_profit = datos.loc[datos["perdidas"], "Profit"].abs().sum()                 # Se agrupa sobre las observaciones que cumplen la condición de pérdidas (profit < 0) y se calcula la suma de las pérdidas absolutas.

# Construcción de un diccionario con los resultados
profitability_summary = pd.DataFrame([{
    "lines": lines,
    "orders": orders,
    "sales": sales,
    "profit": profit,
    "profit_margin": profit_margin,
    "loss_lines": loss_lines,
    "loss_line_rate": loss_line_rate,
    "lost_profit": lost_profit
}])

In [ ]:
with open(SUBMISSION_PROFITABILITY_FILE, "w", encoding="utf-8") as f:
    profitability_summary.to_csv(f, index=False)

In [ ]:
datos["discount_band"] = pd.cut(datos["Discount"],                             # Se crea discount_band, creando los bins según la instrucción del ejercicio
                                bins=[-1 , 0, 0.05, 0.1, np.inf],
                                labels=["0%", "1%-5%", "6%-10%", "más de 10%"]
                                )


datos["perdidas"] = datos["Profit"] < 0                         # Se establece la condición de pérdidas
datos["perdidas_abs"] = datos["Profit"].clip(upper=0).abs()     # Se reemplazan los valores con ganancias por cero para no afectar la sumatoria de perdidas


discount_summary = (                                            # Se agrupa por discount band y se busscan todas las combinaciones posibles 
    datos.groupby("discount_band", observed=False).agg(         # Se aplica agg para realizar calculos al interior de cada banda de descuento
        lines=("Order ID", "size"),                             # Los métodos siez, mean y sum se aplican a cada columna indicada
        sales=("Sales", "sum"),
        profit=("Profit", "sum"),
        loss_line_rate=("perdidas", "mean"),
        lost_profit=("perdidas_abs", "sum")
    )
)

discount_summary["profit_margin"] = discount_summary["profit"] / discount_summary["sales"]  # Se crea profit_margin fuera  de discount summary para aprovechar que ya se tiene el cálculo agregado de profit y sales para definir este término fácilmente
discount_summary = discount_summary.reset_index()                                           # Se resetea el índice

discount_summary = discount_summary[["discount_band", "lines", "sales", "profit", "profit_margin",
                                     "loss_line_rate", "lost_profit"]]                              # Se determina el orden requerido para el futuro archivo. Se usa doble corchete. El primer es el operador de selección, el segundo para crear una lista

discount_summary.to_csv(SUBMISSION_DISCOUNT_FILE, index=False)

In [ ]:
priority_segments = datos.groupby(["Customer Segment", "Product Category"], observed=False).agg(    # Se agrupa por dos categorías
    lines=("Order ID", "size"),
    sales=("Sales", "sum"),
    profit=("Profit", "sum"),
    lost_profit=("perdidas_abs", "sum")
)

priority_segments["profit_margin"] = priority_segments["profit"] / priority_segments["sales"]   # Se calcula profit_margin fuera de la agrupación para aprovechar el cálculo de profit y sales ya agregado
priority_segments = priority_segments.reset_index()

priority_segments = priority_segments[["Customer Segment", "Product Category", "lines",         # Se incluye profit_margin y se ordena la información
                                        "sales", "profit", "profit_margin", "lost_profit"]]

priority_segments = priority_segments[priority_segments["lines"] >= 100]                        # Se establece un filtro sobre lines tal como dice el enunciado
priority_segments = priority_segments.sort_values(by="lost_profit", ascending=False).head(5)    # Se ordena de manera ascendente por lost profit tal como dice el enunciado y se seleccionan las 5 primeras filas

priority_segments.to_csv(SUBMISSION_PRIORITY_FILE, index=False)